# Second Dataset Experiment
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Eric Ly
---


Starter notebook: shared data loading and batch verification are provided; model implementation, training, and results remain to be completed. No results are fabricated.

Retrain the baseline, CNN, LSTM, and Transformer from scratch on a distinct dataset with three seeds each. Fit a separate vocabulary on that dataset's training partition.


## 1. Initial Configuration


### 1.1. Imports, Paths, and Device


In [14]:
# Import required libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Locate the repository from either its root or the notebooks folder
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'src').is_dir() else current_dir.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Place this notebook in the repository notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project modules
from src.config import BATCH_SIZE, TRAINING_SEEDS
from src.data import load_prepared_data
from src.reproducibility import set_seed
from src.text import EncodedTextDataset, make_data_loader

# Use CUDA when it is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the runtime and selected device
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.10
Interpreter: d:\Humber\Advanced Deep Learning\Repos\AIGC5500-Midterm-Project\.venv\Scripts\python.exe
PyTorch: 2.9.1+cu126
Device: cuda
GPU: NVIDIA GeForce GTX 1050 Ti


### 1.2. Experiment Configuration


In [15]:
# Name this experiment and its owner
MODEL_NAME = 'second_dataset'
OWNER = 'Eric Ly'

# Reset randomness before the first run
set_seed(TRAINING_SEEDS[0])

# Set these only after selecting and preparing a different dataset
SECOND_DATASET_ID = 'kaggle'
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'kaggle'

# Stop here until the second dataset is configured
print('Second dataset must be selected and prepared before section 2.')

Second dataset must be selected and prepared before section 2.


## 2. Load Shared Prepared Data


Prepare this dataset separately using the shared cleaning/tokenization policy and save the same artifact schema. Do not reuse the Enron vocabulary. The guard below intentionally stops execution until a distinct dataset is configured.


In [16]:
# Require a prepared directory for a dataset other than Enron
if SECOND_DATASET_ID is None or PREPARED_DATA_DIR is None:
    raise ValueError('Configure a distinct second dataset and its prepared-data directory first.')
if SECOND_DATASET_ID == 'SetFit/enron_spam':
    raise ValueError('The second dataset must differ from Enron.')

# Point results and checkpoints at the second-dataset folders
PREPARED_DATA_DIR = Path(PREPARED_DATA_DIR)
RESULTS_DIR_BASELINE = PROJECT_ROOT / 'results' / 'kaggle' / 'baseline'
RESULTS_DIR_CNN = PROJECT_ROOT / 'results' / 'kaggle' / 'cnn'
RESULTS_DIR_LSTM = PROJECT_ROOT / 'results' / 'kaggle' / 'lstm'
RESULTS_DIR_TRANSFORMER = PROJECT_ROOT / 'results' / 'kaggle' / 'transformer'

CHECKPOINT_DIR_BASELINE = PROJECT_ROOT / 'checkpoints' / 'kaggle' / 'baseline'
CHECKPOINT_DIR_CNN = PROJECT_ROOT / 'checkpoints' / 'kaggle' / 'cnn'
CHECKPOINT_DIR_LSTM = PROJECT_ROOT / 'checkpoints' / 'kaggle' / 'lstm'
CHECKPOINT_DIR_TRANSFORMER = PROJECT_ROOT / 'checkpoints' / 'kaggle' / 'transformer'

# Show the shared training seeds and batch size
print('Training seeds:', TRAINING_SEEDS)
print('Batch size:', BATCH_SIZE)

Training seeds: (42, 123, 2026)
Batch size: 64


In [17]:
# Load the saved splits, vocabulary, and metadata
prepared = load_prepared_data(PREPARED_DATA_DIR)

# Keep the cleaned partitions
splits = prepared['splits']

# Keep the training vocabulary
vocabulary = prepared['vocabulary']

# Keep the preprocessing configuration
metadata = prepared['metadata']

# Use the saved sequence length instead of choosing a new one
MAX_LENGTH = metadata['max_length']

# Confirm the vocabulary size and reserved token IDs
assert len(vocabulary) == metadata['vocabulary_size']
assert vocabulary['<pad>'] == 0 and vocabulary['<unk>'] == 1

# Confirm each split still has the saved number of rows
for name, expected_size in metadata['split_sizes'].items():
    assert len(splits[name]) == expected_size

# Show the loaded split sizes
pd.DataFrame([{'split': name, 'rows': len(part)} for name, part in splits.items()])

,split,rows
0,train,14381
1,validation,3596
2,test,2000


In [18]:
# Confirm the saved metadata names the selected dataset
if metadata['dataset_id'] != SECOND_DATASET_ID:
    raise ValueError('Prepared data does not match the selected second dataset.')

In [19]:
# Print dataset information
print('Vocabulary size:', len(vocabulary))
print('Maximum length:', MAX_LENGTH)
print('Dataset:', metadata['dataset_id'])
print('Dataset revision:', metadata['dataset_revision'])

Vocabulary size: 10000
Maximum length: 256
Dataset: kaggle
Dataset revision: main


### 2.1. Encode Development Partitions


Only training and validation are encoded. Recreate the training loader for each seed; keep validation order fixed.


In [20]:
# Encode train and validation only
encoded_datasets = {
    name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH,
        ) for name in ('train', 'validation')
    }

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Inspect one validation batch
example_batch = next(iter(validation_loader))
for name, tensor in example_batch.items():
    print(name, tuple(tensor.shape))

# Recreate the training loader inside every seeded run, before training.
# Do not encode or evaluate the test partition during development.

tokens (64, 256)
lengths (64,)
labels (64,)


In [21]:
from src.config import (TUNING_SEED, WEIGHT_DECAY, MAX_GRAD_NORM, SELECTION_METRIC)
from src.config import EPOCHS, LEARNING_RATES

VOCAB_SIZE = len(vocabulary)
TRAIN_CONFIG = dict(epochs = 10, lr = 1e-3, weight_decay = 1e-2, grad_clip = 1.0, batch_size = BATCH_SIZE)

## 3 Baseline Model

In [ ]:
from src.baseline_model import BaselineModel

RESULTS_DIR_BASELINE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR_BASELINE.mkdir(parents=True, exist_ok=True)

INPUT_KEY = "tokens"
LABEL_KEY = "labels"
PADDING_IDX = 0

assert INPUT_KEY in example_batch
assert LABEL_KEY in example_batch
assert example_batch[INPUT_KEY].ndim == 2
assert example_batch[LABEL_KEY].ndim == 1
assert set(torch.unique(example_batch[LABEL_KEY]).tolist()).issubset({0, 1})

In [15]:
training_history = []
seed_summaries = []

for seed in TRAINING_SEEDS:

    print(f"\n{'=' * 50}")
    print(f"Training baseline with seed: {seed}")
    print(f"{'=' * 50}")

    set_seed(seed)

    train_loader = make_data_loader(
        encoded_datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=seed
    )

    model = BaselineModel(
        vocab_size=len(vocabulary),
        embedding_dim=64,
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATES[0],
        weight_decay=WEIGHT_DECAY
    )

    best_f1 = -1.0
    best_epoch = 0
    best_checkpoint = CHECKPOINT_DIR_BASELINE / f"best_seed_{seed}.pt"

    # 3. Training epochs
    for epoch in range(1, EPOCHS + 1):

        model.train()
        total_train_loss = 0.0
        total_train_samples = 0

        for batch in train_loader:

            inputs = batch[INPUT_KEY].to(device).long()
            labels = batch[LABEL_KEY].to(device).long()

            optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, labels)

            loss.backward()

            # Limit the gradient norm to help stabilize training
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=MAX_GRAD_NORM
            )

            optimizer.step()

            batch_size_actual = labels.size(0)
            total_train_loss += loss.item() * batch_size_actual
            total_train_samples += batch_size_actual

        train_loss = total_train_loss / max(total_train_samples, 1)

        # 4. Validation
        model.eval()

        total_val_loss = 0.0
        total_val_samples = 0
        correct = 0
        true_positive = 0
        false_positive = 0
        false_negative = 0

        with torch.no_grad():
            for batch in validation_loader:

                inputs = batch[INPUT_KEY].to(device).long()
                labels = batch[LABEL_KEY].to(device).long()

                logits = model(inputs)
                loss = criterion(logits, labels)
                predictions = logits.argmax(dim=1)

                n = labels.size(0)
                total_val_loss += loss.item() * n
                total_val_samples += n
                correct += (predictions == labels).sum().item()

                true_positive += (
                    (predictions == 1) & (labels == 1)
                ).sum().item()

                false_positive += (
                    (predictions == 1) & (labels == 0)
                ).sum().item()

                false_negative += (
                    (predictions == 0) & (labels == 1)
                ).sum().item()

        val_loss = total_val_loss / max(total_val_samples, 1)
        accuracy = correct / max(total_val_samples, 1)

        precision = true_positive / max(
            true_positive + false_positive, 1
        )
        recall = true_positive / max(
            true_positive + false_negative, 1
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        print(
            f"Seed {seed} | Epoch {epoch}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {accuracy:.4f} | "
            f"Val Precision: {precision:.4f} | "
            f"Val Recall: {recall:.4f} | "
            f"Val F1: {f1:.4f}"
        )

        training_history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "validation_accuracy": accuracy,
            "validation_precision": precision,
            "validation_recall": recall,
            "validation_f1": f1
        })

        # Keep the checkpoint with the best validation F1
        if f1 > best_f1:
            best_f1 = f1
            best_epoch = epoch

            torch.save({
                "seed": seed,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "vocab_size": len(vocabulary),
                "embedding_dim": 64,
                "padding_idx": PADDING_IDX,
                "max_length": MAX_LENGTH,
                "validation_f1": best_f1,
                "optimizer": "AdamW",
                "learning_rate": LEARNING_RATES[0],
                "weight_decay": WEIGHT_DECAY,
                "gradient_clip": MAX_GRAD_NORM,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE
            }, best_checkpoint)

    seed_summaries.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "best_validation_f1": best_f1,
        "checkpoint": str(best_checkpoint)
    })



Training baseline with seed: 42
Seed 42 | Epoch 1/10 | Train Loss: 0.5689 | Val Loss: 0.4350 | Val Accuracy: 0.8646 | Val Precision: 0.8762 | Val Recall: 0.8278 | Val F1: 0.8513
Seed 42 | Epoch 2/10 | Train Loss: 0.3202 | Val Loss: 0.2455 | Val Accuracy: 0.9294 | Val Precision: 0.9256 | Val Recall: 0.9234 | Val F1: 0.9245
Seed 42 | Epoch 3/10 | Train Loss: 0.1979 | Val Loss: 0.1778 | Val Accuracy: 0.9472 | Val Precision: 0.9431 | Val Recall: 0.9442 | Val F1: 0.9436
Seed 42 | Epoch 4/10 | Train Loss: 0.1475 | Val Loss: 0.1478 | Val Accuracy: 0.9538 | Val Precision: 0.9507 | Val Recall: 0.9507 | Val F1: 0.9507
Seed 42 | Epoch 5/10 | Train Loss: 0.1194 | Val Loss: 0.1315 | Val Accuracy: 0.9588 | Val Precision: 0.9555 | Val Recall: 0.9567 | Val F1: 0.9561
Seed 42 | Epoch 6/10 | Train Loss: 0.1008 | Val Loss: 0.1223 | Val Accuracy: 0.9616 | Val Precision: 0.9629 | Val Recall: 0.9549 | Val F1: 0.9589
Seed 42 | Epoch 7/10 | Train Loss: 0.0870 | Val Loss: 0.1155 | Val Accuracy: 0.9650 | Val P

In [16]:
history_df = pd.DataFrame(training_history)
summary_df = pd.DataFrame(seed_summaries)

history_path = RESULTS_DIR_BASELINE / "training_history.csv"
summary_path = RESULTS_DIR_BASELINE / "validation_summary.csv"

history_df.to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)

# Mean and sample standard deviation across the three seeds
mean_f1 = summary_df["best_validation_f1"].mean()
std_f1 = summary_df["best_validation_f1"].std(ddof=1)

print("\nTraining completed.")
print("\nBest validation results:")
display(summary_df)

print(f"Mean best validation F1: {mean_f1:.6f}")
print(f"Sample standard deviation: {std_f1:.6f}")

print("\nSaved files:")
print(history_path)
print(summary_path)
print("Checkpoints:", CHECKPOINT_DIR_BASELINE)


Training completed.

Best validation results:


,seed,best_epoch,best_validation_f1,checkpoint
0,42,8,0.965619,C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midte...
1,123,9,0.962169,C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midte...
2,2026,10,0.963805,C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midte...


Mean best validation F1: 0.963864
Sample standard deviation: 0.001726

Saved files:
C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midterm-Project\results\kaggle\baseline\training_history.csv
C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midterm-Project\results\kaggle\baseline\validation_summary.csv
Checkpoints: C:\Users\Eric\Desktop\AIGC 5500\AIGC5500-Midterm-Project\checkpoints\kaggle\baseline


In [17]:
from src.evaluation import classification_metrics

TEST_FILE = RESULTS_DIR_BASELINE / 'test_per_seed.csv'
ALLOW_TEST_RERUN = False
if TEST_FILE.exists() and not ALLOW_TEST_RERUN:
    raise RuntimeError('Test results already exist. The test split is scored once; do not rerun after seeing it.')

# Encode the test split
test_dataset = EncodedTextDataset(texts = splits['test']['text'], labels = splits['test']['label'],
                                  vocabulary = vocabulary, max_length = MAX_LENGTH)
test_loader = make_data_loader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Score every frozen model once
test_rows = []
for seed in TRAINING_SEEDS:
    checkpoint = torch.load(CHECKPOINT_DIR_BASELINE / f'best_seed_{seed}.pt', map_location = device, weights_only = True)

    model = BaselineModel(vocab_size = checkpoint['vocab_size'], embedding_dim = checkpoint['embedding_dim'],
                          padding_idx = checkpoint['padding_idx']).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    y_true, y_pred = [], []
    with torch.inference_mode():
        for batch in test_loader:
            predictions = model(batch['tokens'].to(device)).argmax(dim = 1)
            y_true.extend(batch['labels'].tolist())
            y_pred.extend(predictions.cpu().tolist())

    result = classification_metrics(y_true, y_pred)
    test_rows.append({'variant': 'Baseline', 'seed': seed,
                      'test_accuracy': result['accuracy'], 'test_f1': result['spam_f1']})

test_results = pd.DataFrame(test_rows)
test_results.to_csv(TEST_FILE, index = False)
test_results.round(4)

,variant,seed,test_accuracy,test_f1
0,Baseline,42,0.9650,0.9628
1,Baseline,123,0.9695,0.9675
2,Baseline,2026,0.9685,0.9662


In [18]:
test_summary = test_results.groupby('variant')[['test_accuracy', 'test_f1']].agg(['mean', 'std'])
test_summary.to_csv(RESULTS_DIR_BASELINE / 'test_summary.csv')
test_summary.round(4)

test_accuracy         test_f1        
                  mean     std    mean     std
variant                                       
Baseline        0.9677  0.0024  0.9655  0.0024

## 4. Convolutional Model

In [ ]:
from src.cnn_model import (TextCNN, build_model, count_parameters, evaluate, train_one_run,
                           load_checkpoint, measure_cost, hardware_description)
    
MODEL_CONFIG_CNN = dict(embed_dim = 64, num_filters = 100, dropout = 0.1)

VARIANTS_CNN = {
    'CNN (full)': dict(kernel_sizes = (3, 5, 7)),
}

VARIANT_SLUG_CNN = {name: name.lower().replace(' ', '_').replace(':', '').replace('(', '').replace(')', '') for name in VARIANTS_CNN}

In [9]:
print('Model config:', MODEL_CONFIG_CNN)
print('Train config:', TRAIN_CONFIG)
print('Hardware:', hardware_description(device))

parameter_table = pd.DataFrame({name: {'parameters': count_parameters(build_model(VOCAB_SIZE, MODEL_CONFIG_CNN, cfg))}
                                for name, cfg in VARIANTS_CNN.items()}).T
parameter_table

Model config: {'embed_dim': 64, 'num_filters': 100, 'dropout': 0.1}
Train config: {'epochs': 10, 'lr': 0.001, 'weight_decay': 0.01, 'grad_clip': 1.0, 'batch_size': 64}
Hardware: CPU: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel (16 threads)


,parameters
CNN (full),736902


In [10]:
# Quick check: one batch in, one pair of scores (ham, spam) per email out
check_model = build_model(VOCAB_SIZE, MODEL_CONFIG_CNN, VARIANTS_CNN['CNN (full)']).to(device)
with torch.no_grad():
    logits = check_model(example_batch['tokens'].to(device))
print('Input tokens:', tuple(example_batch['tokens'].shape), '-> output scores:', tuple(logits.shape))
assert logits.shape == (example_batch['tokens'].shape[0], 2)
del check_model, logits

Input tokens: (64, 256) -> output scores: (64, 2)


In [11]:
RESULTS_DIR_CNN.mkdir(parents = True, exist_ok = True)
CHECKPOINT_DIR_CNN.mkdir(parents = True, exist_ok = True)

validation_rows, histories = [], {}
for variant, variant_config in VARIANTS_CNN.items():
    for seed in TRAINING_SEEDS:
        checkpoint = CHECKPOINT_DIR_CNN / f'{VARIANT_SLUG_CNN[variant]}_seed{seed}.pt'
        model, history, best_epoch, seconds = train_one_run(
            MODEL_CONFIG_CNN, variant_config, seed, encoded_datasets['train'], validation_loader,
            VOCAB_SIZE, device, TRAIN_CONFIG, checkpoint)

        # Record the scores from the best epoch
        best = history.loc[history['epoch'] == best_epoch].iloc[0]
        histories[(variant, seed)] = history
        validation_rows.append({'variant': variant, 'seed': seed, 'best_epoch': best_epoch,
                                'val_accuracy': best['val_accuracy'], 'val_f1': best['val_f1'],
                                'val_loss': best['val_loss'], 'train_seconds': round(seconds, 1)})
        print(f'{variant:26s} seed {seed:5d}  best epoch {best_epoch:2d}  '
              f'val acc {best["val_accuracy"]:.4f}  F1 {best["val_f1"]:.4f}  ({seconds:.0f}s)')

validation_results = pd.DataFrame(validation_rows)
validation_results.to_csv(RESULTS_DIR_CNN / 'validation_per_seed.csv', index = False)
validation_results.round(4)

CNN (full)                 seed    42  best epoch 10  val acc 0.9644  F1 0.9622  (59s)
CNN (full)                 seed   123  best epoch 10  val acc 0.9580  F1 0.9558  (62s)
CNN (full)                 seed  2026  best epoch 10  val acc 0.9630  F1 0.9610  (62s)


,variant,seed,best_epoch,val_accuracy,val_f1,val_loss,train_seconds
0,CNN (full),42,10,0.9644,0.9622,0.1221,59.3
1,CNN (full),123,10,0.9580,0.9558,0.1554,62.1
2,CNN (full),2026,10,0.9630,0.9610,0.1305,61.5


In [12]:
# Mean and spread over the 3 seeds
validation_summary = validation_results.groupby('variant')[['val_accuracy', 'val_f1']].agg(['mean', 'std'])
validation_summary.round(4)

val_accuracy          val_f1        
                   mean     std    mean     std
variant                                        
CNN (full)       0.9618  0.0034  0.9597  0.0034

In [16]:
TEST_FILE = RESULTS_DIR_CNN / 'test_per_seed.csv'
ALLOW_TEST_RERUN = False   # keep False: the test split is scored only once
if TEST_FILE.exists() and not ALLOW_TEST_RERUN:
    raise RuntimeError('Test results already exist. The test split is scored once; do not rerun after seeing it.')

# The only place the test split is encoded
test_dataset = EncodedTextDataset(texts = splits['test']['text'], labels = splits['test']['label'],
                                  vocabulary = vocabulary, max_length = MAX_LENGTH)
test_loader = make_data_loader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Score every frozen model once
test_rows = []
for variant, variant_config in VARIANTS_CNN.items():
    for seed in TRAINING_SEEDS:
        checkpoint = CHECKPOINT_DIR_CNN / f'{VARIANT_SLUG_CNN[variant]}_seed{seed}.pt'
        model = load_checkpoint(checkpoint, VOCAB_SIZE, MODEL_CONFIG_CNN, variant_config, device)
        result = evaluate(model, test_loader, device)
        test_rows.append({'variant': variant, 'seed': seed, 'test_accuracy': result['accuracy'], 'test_f1': result['f1']})

test_results = pd.DataFrame(test_rows)
test_results.to_csv(TEST_FILE, index = False)
test_results.round(4)

,variant,seed,test_accuracy,test_f1
0,CNN (full),42,0.9605,0.9580
1,CNN (full),123,0.9530,0.9512
2,CNN (full),2026,0.9650,0.9630


In [17]:
# Mean and spread over the 3 seeds
test_summary = test_results.groupby('variant')[['test_accuracy', 'test_f1']].agg(['mean', 'std'])
test_summary.to_csv(RESULTS_DIR_CNN / 'test_summary.csv')
test_summary.round(4)

test_accuracy         test_f1        
                    mean     std    mean     std
variant                                         
CNN (full)        0.9595  0.0061  0.9574  0.0059

## 5. Recurrent Model

In [19]:
from src.lstm_model import LSTMClassifier
from src.evaluation import classification_metrics, evaluate_model
from src.training import fit_model

In [20]:
MODEL_CONFIG_LSTM = {'embed_dim': 64, 'hidden_dim': 128, 'dropout': 0.1}

model = LSTMClassifier(vocab_size = len(vocabulary), **MODEL_CONFIG_LSTM, forget_bias = 0.0).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print(f'Trainable parameters: {parameter_count:,}')

Trainable parameters: 739,586


In [22]:
# Fixed learning rate agreed by the team
SELECTED_LR = 1e-3 

# Create the directories if they don't exist
RESULTS_DIR_LSTM.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR_LSTM.mkdir(parents=True, exist_ok=True)

# Initialize the lists
validation_rows = []
experiment_histories = {}

# Train the models
for variant, forget_bias in [('LSTM', 0.0)]:
    # Train the models for each seed
    for seed in TRAINING_SEEDS:
        # Define the run name
        run_name = f'e{EPOCHS}_{SELECTION_METRIC}_lr{SELECTED_LR:g}_seed{seed}_fb{forget_bias:g}'

        # Define the checkpoint and history paths
        checkpoint_path = CHECKPOINT_DIR_LSTM / f'{run_name}.pt'
        history_path = RESULTS_DIR_LSTM / f'{run_name}_history.csv'

        # Train only when neither output exists
        if not checkpoint_path.exists() and not history_path.exists():
            print(f'\n{variant} | seed {seed}')

            completed_run = fit_model(model_factory = lambda: LSTMClassifier(
                    vocab_size = len(vocabulary),
                    **MODEL_CONFIG_LSTM,
                    forget_bias = forget_bias,
                ),
                train_dataset = encoded_datasets['train'],
                validation_loader = validation_loader,
                device = device,
                seed =seed,
                epochs = EPOCHS,
                learning_rate = SELECTED_LR,
                weight_decay = WEIGHT_DECAY,
                max_grad_norm = MAX_GRAD_NORM,
                batch_size = BATCH_SIZE,
                checkpoint_path = checkpoint_path,
                history_path = history_path,
                selection_metric = SELECTION_METRIC,
            )
            # Delete the completed run
            del completed_run

        # Stop if an earlier run left incomplete outputs.
        if not checkpoint_path.exists() or not history_path.exists():
            raise RuntimeError(f'Incomplete outputs for {run_name}. Archive the incomplete files before rerunning.')

        # Load the checkpoint and history
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        history = pd.read_csv(history_path)

        # Verify that saved outputs match the final protocol.
        expected_settings = {
            'seed': seed,
            'epochs_budget': 10,
            'learning_rate': SELECTED_LR,
            'selection_metric': SELECTION_METRIC,
            'batch_size': BATCH_SIZE,
            'weight_decay': WEIGHT_DECAY,
            'max_grad_norm': MAX_GRAD_NORM,
        }

        # Verify that the saved outputs match the current protocol
        for name, expected in expected_settings.items():
            if checkpoint.get(name) != expected:
                raise ValueError(f'{run_name}: saved {name} does not match the current protocol.')

        # Verify that the history contains all epochs
        if history['epoch'].tolist() != list(range(1, EPOCHS + 1)):
            raise RuntimeError(f'{run_name}: history does not contain all epochs.')

        experiment_histories[(variant, seed)] = history

        # Append the validation row
        validation_rows.append({
            'variant': variant,
            'seed': seed,
            'forget_bias': forget_bias,
            'learning_rate': SELECTED_LR,
            'epochs_budget': 10,
            'selection_metric': SELECTION_METRIC,
            'best_epoch': checkpoint['epoch'],
            'checkpoint_path': (checkpoint_path.relative_to(PROJECT_ROOT).as_posix()),
            'history_path': (history_path.relative_to(PROJECT_ROOT).as_posix()),
            **{f'val_{name}': value for name, value in checkpoint['validation_metrics'].items()},
        })

        # Save the validation rows
        pd.DataFrame(validation_rows).to_csv(RESULTS_DIR_LSTM / 'validation_per_seed.csv', index=False)

        del checkpoint

validation_results = pd.DataFrame(validation_rows)
validation_results


LSTM | seed 123
Epoch 01/10 | train loss 0.4601 | val loss 0.3495 | val spam_f1 0.8507
Epoch 02/10 | train loss 0.2711 | val loss 0.3334 | val spam_f1 0.8773
Epoch 03/10 | train loss 0.1952 | val loss 0.2533 | val spam_f1 0.8983
Epoch 04/10 | train loss 0.1562 | val loss 0.2497 | val spam_f1 0.9016
Epoch 05/10 | train loss 0.1176 | val loss 0.2258 | val spam_f1 0.9135
Epoch 06/10 | train loss 0.0984 | val loss 0.1906 | val spam_f1 0.9344
Epoch 07/10 | train loss 0.0732 | val loss 0.2078 | val spam_f1 0.9392
Epoch 08/10 | train loss 0.0551 | val loss 0.2242 | val spam_f1 0.9285
Epoch 09/10 | train loss 0.0423 | val loss 0.2311 | val spam_f1 0.9350
Epoch 10/10 | train loss 0.0357 | val loss 0.2595 | val spam_f1 0.9359

LSTM | seed 2026
Epoch 01/10 | train loss 0.4591 | val loss 0.3240 | val spam_f1 0.8643
Epoch 02/10 | train loss 0.2752 | val loss 0.2596 | val spam_f1 0.8980
Epoch 03/10 | train loss 0.1973 | val loss 0.2473 | val spam_f1 0.9019
Epoch 04/10 | train loss 0.1522 | val loss

,variant,seed,forget_bias,learning_rate,epochs_budget,selection_metric,best_epoch,checkpoint_path,history_path,val_accuracy,val_macro_f1,val_spam_f1,val_spam_precision,val_spam_recall,val_loss
0,LSTM,42,0.0,0.001,10,spam_f1,9,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,results/kaggle/lstm/e10_spam_f1_lr0.001_seed42...,0.945495,0.945275,0.941805,0.941805,0.941805,0.219657
1,LSTM,123,0.0,0.001,10,spam_f1,7,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,results/kaggle/lstm/e10_spam_f1_lr0.001_seed12...,0.943826,0.943503,0.939230,0.951829,0.926960,0.207778
2,LSTM,2026,0.0,0.001,10,spam_f1,9,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,results/kaggle/lstm/e10_spam_f1_lr0.001_seed20...,0.946607,0.946414,0.943195,0.939858,0.946556,0.194616


In [24]:
# Define the validation summary
validation_summary = (validation_results.groupby("variant")[["val_accuracy", "val_macro_f1", "val_spam_f1"]].agg(["mean", "std"]))
validation_summary.to_csv(RESULTS_DIR_LSTM / "validation_summary.csv")
validation_summary.round(4)

val_accuracy         val_macro_f1         val_spam_f1       
                mean     std         mean     std        mean    std
variant                                                             
LSTM          0.9453  0.0014       0.9451  0.0015      0.9414  0.002

In [25]:
# Encode the held-out test partition using the training vocabulary.
test_dataset = EncodedTextDataset(texts = splits['test']['text'], labels = splits['test']['label'], vocabulary = vocabulary, max_length = MAX_LENGTH)

# Define the test loader using the test dataset
test_loader = make_data_loader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, seed = TUNING_SEED)

# Define the test rows list
test_rows = []

for _, row in validation_results.iterrows():
    # Recreate the architecture associated with this checkpoint
    test_model = LSTMClassifier(vocab_size = len(vocabulary), **MODEL_CONFIG_LSTM, forget_bias = float(row['forget_bias'])).to(device)

    checkpoint_path = PROJECT_ROOT / row['checkpoint_path']
    checkpoint = torch.load(checkpoint_path, map_location = device, weights_only = True)
    test_model.load_state_dict(checkpoint['model_state_dict'])

    # Evaluate without updating model weights.
    metrics = evaluate_model(test_model, test_loader, device)

    # Append the test row to the test rows list
    test_rows.append({
        'variant': row['variant'],
        'seed': int(row['seed']),
        'forget_bias': float(row['forget_bias']),
        'learning_rate': float(row['learning_rate']),
        'best_epoch': int(row['best_epoch']),
        'checkpoint_path': row['checkpoint_path'],
        **{f'test_{name}': value for name, value in metrics.items()},
    })

    print(f"{row['variant']} | seed {int(row['seed'])} | test macro F1: {metrics['macro_f1']:.4f}")

    del test_model, checkpoint

# Save the test results
test_results = pd.DataFrame(test_rows)
test_results.to_csv(RESULTS_DIR_LSTM / 'test_per_seed.csv', index = False)
test_results

LSTM | seed 42 | test macro F1: 0.9418
LSTM | seed 123 | test macro F1: 0.9412
LSTM | seed 2026 | test macro F1: 0.9489


,variant,seed,forget_bias,learning_rate,best_epoch,checkpoint_path,test_accuracy,test_macro_f1,test_spam_f1,test_spam_precision,test_spam_recall,test_loss
0,LSTM,42,0.0,0.001,9,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,0.9420,0.941811,0.938494,0.932561,0.944504,0.198267
1,LSTM,123,0.0,0.001,7,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,0.9415,0.941224,0.937198,0.942765,0.931697,0.200147
2,LSTM,2026,0.0,0.001,9,checkpoints/kaggle/lstm/e10_spam_f1_lr0.001_se...,0.9490,0.948882,0.946429,0.931748,0.961580,0.188406


In [26]:
# Summarize performance across the three training seeds.
test_metric_columns = ['test_accuracy', 'test_macro_f1', 'test_spam_f1', 'test_spam_precision', 'test_spam_recall']
test_summary = test_results.groupby('variant')[test_metric_columns].agg(['mean', 'std'])
test_summary.to_csv(RESULTS_DIR_LSTM / 'test_summary.csv')
test_summary.round(4)

test_accuracy         test_macro_f1         test_spam_f1         \
                 mean     std          mean     std         mean    std   
variant                                                                   
LSTM           0.9442  0.0042         0.944  0.0043       0.9407  0.005   

        test_spam_precision         test_spam_recall         
                       mean     std             mean    std  
variant                                                      
LSTM                 0.9357  0.0061           0.9459  0.015

## 6. Attention Model

In [22]:
# Model code lives in src/transformer_model.py (written for this project, trained from scratch)
import src.transformer_model
from src.transformer_model import count_parameters, evaluate, hardware_description, load_checkpoint

VOCAB_SIZE = len(vocabulary)

# Architecture (same for every variant except the one ablated component)
MODEL_CONFIG_TRANSFORMER = dict(embed_dim = 64, num_heads = 4, num_layers = 2, ff_dim = 128, dropout = 0.1)

# One ablation: remove positional information. Everything else is held fixed.
VARIANTS_TRANSFORMER = {
    'Transformer (full)': dict(use_positions = True),
}
VARIANT_SLUG_TRANSFORMER = {name: name.lower().replace(' ', '_').replace(':', '').replace('(', '').replace(')', '') for name in VARIANTS_TRANSFORMER}

print('Model config:', MODEL_CONFIG_TRANSFORMER)
print('Train config:', TRAIN_CONFIG)
print('Hardware:', hardware_description(device))

Model config: {'embed_dim': 64, 'num_heads': 4, 'num_layers': 2, 'ff_dim': 128, 'dropout': 0.1}
Train config: {'epochs': 10, 'lr': 0.001, 'weight_decay': 0.01, 'grad_clip': 1.0, 'batch_size': 64}
Hardware: GPU: NVIDIA GeForce GTX 1050 Ti


In [23]:
# Parameter counts (compare with the baseline, CNN and LSTM: they should be within about 2x, or the gap explained)
pd.DataFrame({name: {'parameters': count_parameters(src.transformer_model.build_model(VOCAB_SIZE, MAX_LENGTH, MODEL_CONFIG_TRANSFORMER, cfg))}
              for name, cfg in VARIANTS_TRANSFORMER.items()}).T

,parameters
Transformer (full),723586


In [24]:
RESULTS_DIR_TRANSFORMER.mkdir(parents = True, exist_ok = True)
CHECKPOINT_DIR_TRANSFORMER.mkdir(parents = True, exist_ok = True)

validation_rows, histories = [], {}
for variant, variant_config in VARIANTS_TRANSFORMER.items():
    for seed in TRAINING_SEEDS:
        checkpoint = CHECKPOINT_DIR_TRANSFORMER / f'{VARIANT_SLUG_TRANSFORMER[variant]}_seed{seed}.pt'
        model, history, best_epoch, seconds = src.transformer_model.train_one_run(
            MODEL_CONFIG_TRANSFORMER, variant_config, seed, encoded_datasets['train'], validation_loader,
            VOCAB_SIZE, MAX_LENGTH, device, TRAIN_CONFIG, checkpoint)
        best = history.loc[history['epoch'] == best_epoch].iloc[0]
        histories[(variant, seed)] = history
        validation_rows.append({'variant': variant, 'seed': seed, 'best_epoch': best_epoch,
                                'val_accuracy': best['val_accuracy'], 'val_f1': best['val_f1'],
                                'val_loss': best['val_loss'], 'train_seconds': round(seconds, 1)})
        print(f'{variant:24s} seed {seed:5d}  best epoch {best_epoch:2d}  val acc {best["val_accuracy"]:.4f}  F1 {best["val_f1"]:.4f}  ({seconds:.0f}s)')

validation_results = pd.DataFrame(validation_rows)
validation_results.to_csv(RESULTS_DIR_TRANSFORMER / 'validation_per_seed.csv', index = False)
validation_results.round(4)

Transformer (full)       seed    42  best epoch  9  val acc 0.9622  F1 0.9595  (179s)
Transformer (full)       seed   123  best epoch  9  val acc 0.9619  F1 0.9591  (177s)
Transformer (full)       seed  2026  best epoch 10  val acc 0.9602  F1 0.9571  (177s)


,variant,seed,best_epoch,val_accuracy,val_f1,val_loss,train_seconds
0,Transformer (full),42,9,0.9622,0.9595,0.1443,178.6
1,Transformer (full),123,9,0.9619,0.9591,0.1565,177.0
2,Transformer (full),2026,10,0.9602,0.9571,0.1704,176.7


In [25]:
# Mean and spread across seeds (sample standard deviation, n = 3)
validation_summary = validation_results.groupby('variant')[['val_accuracy', 'val_f1']].agg(['mean', 'std'])
validation_summary.round(4)

val_accuracy          val_f1        
                           mean     std    mean     std
variant                                                
Transformer (full)       0.9614  0.0011  0.9586  0.0013

In [26]:
TEST_FILE = RESULTS_DIR_TRANSFORMER / 'test_per_seed.csv'
ALLOW_TEST_RERUN = False   # leave False: the test split is scored once
if TEST_FILE.exists() and not ALLOW_TEST_RERUN:
    raise RuntimeError('Test results already exist. The test split is scored once; do not rerun after seeing it.')

# Encode the test partition here and nowhere else
test_dataset = EncodedTextDataset(texts = splits['test']['text'], labels = splits['test']['label'],
                                  vocabulary = vocabulary, max_length = MAX_LENGTH)
test_loader = make_data_loader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

test_rows = []
for variant, variant_config in VARIANTS_TRANSFORMER.items():
    for seed in TRAINING_SEEDS:
        checkpoint = CHECKPOINT_DIR_TRANSFORMER / f'{VARIANT_SLUG_TRANSFORMER[variant]}_seed{seed}.pt'
        model = load_checkpoint(checkpoint, VOCAB_SIZE, MAX_LENGTH, MODEL_CONFIG_TRANSFORMER, variant_config, device)
        result = evaluate(model, test_loader, device)
        test_rows.append({'variant': variant, 'seed': seed, 'test_accuracy': result['accuracy'], 'test_f1': result['f1']})

test_results = pd.DataFrame(test_rows)
test_results.to_csv(TEST_FILE, index = False)
test_results.round(4)

,variant,seed,test_accuracy,test_f1
0,Transformer (full),42,0.9610,0.9585
1,Transformer (full),123,0.9675,0.9652
2,Transformer (full),2026,0.9580,0.9548


In [27]:
# Final table for the report: every seed, mean and spread
test_summary = test_results.groupby('variant')[['test_accuracy', 'test_f1']].agg(['mean', 'std'])
test_summary.to_csv(RESULTS_DIR_TRANSFORMER / 'test_summary.csv')
test_summary.round(4)

test_accuracy         test_f1        
                            mean     std    mean     std
variant                                                 
Transformer (full)        0.9622  0.0049  0.9595  0.0052